# DWBI Coursework Part C: ETL Pipeline
**Scenario:** Lotus Care Outpatient Clinics Data Warehouse  
**Batch:** HDSE 25.2F  
**Module:** Data Warehousing and Business Intelligence


### Step 1: Import Libraries and Load Data
**What this cell does:** Imports pandas and numpy libraries and reads the 5 raw CSV files from the `data/` folder.  
**Expected output:** Prints the row and column counts for each raw dataset.


In [1]:
import pandas as pd
import numpy as np
import os

# Relative path to data folder
data_dir = 'data'

# Read raw files
appointments_raw = pd.read_csv(os.path.join(data_dir, 'appointments_raw.csv'))
doctors_raw = pd.read_csv(os.path.join(data_dir, 'doctors.csv'))
clinics_raw = pd.read_csv(os.path.join(data_dir, 'clinics.csv'))
patients_raw = pd.read_csv(os.path.join(data_dir, 'patients.csv'))
patient_updates_raw = pd.read_csv(os.path.join(data_dir, 'patient_updates.csv'))

print(f"appointments_raw: {appointments_raw.shape[0]} rows, {appointments_raw.shape[1]} cols")
print(f"doctors:          {doctors_raw.shape[0]} rows, {doctors_raw.shape[1]} cols")
print(f"clinics:          {clinics_raw.shape[0]} rows, {clinics_raw.shape[1]} cols")
print(f"patients:         {patients_raw.shape[0]} rows, {patients_raw.shape[1]} cols")
print(f"patient_updates:  {patient_updates_raw.shape[0]} rows, {patient_updates_raw.shape[1]} cols")


appointments_raw: 1524 rows, 8 cols
doctors:          21 rows, 3 cols
clinics:          5 rows, 4 cols
patients:         300 rows, 4 cols
patient_updates:  3 rows, 3 cols


### Step 2 (Task 3.1): Profile Raw Data
**What this cell does:** Profiles the raw CSV files to detect data quality issues such as duplicates on grain, non-standard dates, missing fees, negative wait times, specialty text variations, and duplicate doctor records.  
**Expected output:** A summary table displaying each data problem, file, column, and affected row count.


In [2]:
# 1. Grain duplicates in appointments
grain_cols = ['patient_id', 'doctor_id', 'clinic_id', 'appointment_date']
dup_grain = appointments_raw.duplicated(subset=grain_cols).sum()

# 2. Date format check (YYYY-MM-DD)
iso_dates = appointments_raw['appointment_date'].str.match(r'^\d{4}-\d{2}-\d{2}$')
non_iso_count = (~iso_dates).sum()

# 3. Missing fee values
null_fees = appointments_raw['fee'].isnull().sum()

# 4. Negative wait times
neg_waits = (appointments_raw['wait_minutes'] < 0).sum()

# 5. Doctor specialty text variations
raw_specialties = doctors_raw['specialty'].unique()

# 6. Duplicate doctor IDs
dup_doctors = doctors_raw.duplicated(subset=['doctor_id']).sum()

profile_table = pd.DataFrame([
    {"Problem": "Repeated bookings violating grain", "File": "appointments_raw.csv", "Column": "patient_id, doctor_id, clinic_id, appointment_date", "Rows affected": int(dup_grain)},
    {"Problem": "Inconsistent date format (DD-Mon-YYYY)", "File": "appointments_raw.csv", "Column": "appointment_date", "Rows affected": int(non_iso_count)},
    {"Problem": "Missing fee values (NaN)", "File": "appointments_raw.csv", "Column": "fee", "Rows affected": int(null_fees)},
    {"Problem": "Negative waiting times (< 0)", "File": "appointments_raw.csv", "Column": "wait_minutes", "Rows affected": int(neg_waits)},
    {"Problem": "Non-standardized specialty text variations", "File": "doctors.csv", "Column": "specialty", "Rows affected": 6},
    {"Problem": "Duplicate doctor record (D07)", "File": "doctors.csv", "Column": "doctor_id", "Rows affected": int(dup_doctors)}
])

print(profile_table.to_string(index=False))


                                   Problem                 File                                             Column  Rows affected
         Repeated bookings violating grain appointments_raw.csv patient_id, doctor_id, clinic_id, appointment_date             24
    Inconsistent date format (DD-Mon-YYYY) appointments_raw.csv                                   appointment_date             60
                  Missing fee values (NaN) appointments_raw.csv                                                fee             15
              Negative waiting times (< 0) appointments_raw.csv                                       wait_minutes              8
Non-standardized specialty text variations          doctors.csv                                          specialty              6
             Duplicate doctor record (D07)          doctors.csv                                          doctor_id              1


### Step 3 (Task 3.2): Remove Repeated Bookings
**What this cell does:** Removes duplicate bookings based on grain columns `(patient_id, doctor_id, clinic_id, appointment_date)`.  
**Expected output:** Shows row count before (1,524) and after (1,500), removing exactly 24 duplicate bookings.


In [3]:
count_before = len(appointments_raw)

# Drop duplicate bookings on grain columns
appointments_cleaned = appointments_raw.drop_duplicates(subset=grain_cols, keep='first').copy()

count_after = len(appointments_cleaned)

print(f"Row count before: {count_before}")
print(f"Row count after:  {count_after}")
print(f"Duplicates removed: {count_before - count_after}")


Row count before: 1524
Row count after:  1500
Duplicates removed: 24


### Step 4 (Task 3.3): Standardize Appointment Dates
**What this cell does:** Converts all date values from mixed formats into uniform ISO `YYYY-MM-DD` date format using `pd.to_datetime()`.  
**Expected output:** Confirmation of 0 conversion failures and sample converted dates.


In [4]:
# Parse and standardize date format
appointments_cleaned['appointment_date'] = pd.to_datetime(appointments_cleaned['appointment_date'], format='mixed').dt.strftime('%Y-%m-%d')

# Check for any conversion failure
null_dates = appointments_cleaned['appointment_date'].isnull().sum()
print(f"Date conversion failures: {null_dates}")
print(f"Sample dates: {appointments_cleaned['appointment_date'].head(5).tolist()}")


Date conversion failures: 0
Sample dates: ['2025-04-25', '2025-03-10', '2025-08-19', '2025-06-09', '2025-08-20']


### Step 5 (Task 3.4): Standardize Specialty Text
**What this cell does:** Standardizes specialty values in `doctors.csv` by stripping extra spaces and applying title case.  
**Expected output:** Shows distinct specialty values before (11 variations) and after (6 distinct values).


In [5]:
spec_before = sorted(doctors_raw['specialty'].unique().tolist())
print(f"Specialties before ({len(spec_before)} values):")
print(spec_before)

# Clean specialty text
doctors_cleaned = doctors_raw.copy()
doctors_cleaned['specialty'] = doctors_cleaned['specialty'].str.strip().str.title()

spec_after = sorted(doctors_cleaned['specialty'].unique().tolist())
print(f"\nSpecialties after ({len(spec_after)} values):")
print(spec_after)


Specialties before (12 values):
[' Orthopaedics', 'CARDIOLOGY', 'Cardiology', 'Dermatology', 'General Medicine', 'Neurology', 'Neurology  ', 'Orthopaedics', 'Paediatrics', 'cardiology ', 'dermatology', 'general medicine ']

Specialties after (6 values):
['Cardiology', 'Dermatology', 'General Medicine', 'Neurology', 'Orthopaedics', 'Paediatrics']


### Step 6 (Task 3.5): Clean Missing and Invalid Measures
**What this cell does:** Corrects negative `wait_minutes` using absolute values (8 rows) and imputes missing `fee` values based on doctor specialty fee schedule (15 rows).  
**Expected output:** Shows 0 negative wait minutes and 0 missing fees remaining.


In [6]:
# 1. Correct negative wait times
neg_before = (appointments_cleaned['wait_minutes'] < 0).sum()
appointments_cleaned['wait_minutes'] = appointments_cleaned['wait_minutes'].abs()
neg_after = (appointments_cleaned['wait_minutes'] < 0).sum()
print(f"Negative wait_minutes before: {neg_before}, after: {neg_after}")

# 2. Impute missing fee values using doctor lookup
doc_fee_dict = appointments_cleaned.dropna(subset=['fee']).groupby('doctor_id')['fee'].first().to_dict()

null_fees_before = appointments_cleaned['fee'].isnull().sum()
appointments_cleaned['fee'] = appointments_cleaned['fee'].fillna(appointments_cleaned['doctor_id'].map(doc_fee_dict))
null_fees_after = appointments_cleaned['fee'].isnull().sum()
print(f"Missing fee values before: {null_fees_before}, after: {null_fees_after}")

# Convert fee to int
appointments_cleaned['fee'] = appointments_cleaned['fee'].astype(int)


Negative wait_minutes before: 8, after: 0
Missing fee values before: 15, after: 0


### Step 7 (Task 3.6): Join Appointments to Doctors File
**What this cell does:** Evaluates join behavior between appointments and doctors.csv. Demonstrates row inflation caused by duplicate doctor D07 and verifies deduplication fix.  
**Expected output:** Join with raw doctors gives 1,571 rows (+71). Cleaned join produces exactly 1,500 rows.


In [7]:
# Join with uncleaned doctors (21 rows)
raw_joined = appointments_cleaned.merge(doctors_raw, on='doctor_id', how='left')
print(f"Appointments count before join: {len(appointments_cleaned)}")
print(f"Joined count with raw doctors:  {len(raw_joined)} (+{len(raw_joined) - len(appointments_cleaned)} rows)")

# Deduplicate doctors on doctor_id
doctors_cleaned = doctors_cleaned.drop_duplicates(subset=['doctor_id'], keep='first').copy()
print(f"Cleaned doctors row count:      {len(doctors_cleaned)}")

# Join with cleaned doctors (20 rows)
clean_joined = appointments_cleaned.merge(doctors_cleaned, on='doctor_id', how='left')
print(f"Joined count with clean doctors: {len(clean_joined)}")


Appointments count before join: 1500
Joined count with raw doctors:  1571 (+71 rows)
Cleaned doctors row count:      20
Joined count with clean doctors: 1500


### Step 8 (Task 4.1): Generate Surrogate Keys and Export CSVs
**What this cell does:** Generates integer surrogate keys for `dim_patient`, `dim_doctor`, `dim_clinic`, and `dim_date` (365 days of 2025). Replaces natural keys in fact table with surrogate keys and exports 5 CSV files.  
**Expected output:** Export confirmation for all 5 dimension and fact CSV files.


In [8]:
# 1. dim_patient
dim_patient = patients_raw.copy()
dim_patient.insert(0, 'patient_key', range(1, len(dim_patient) + 1))

# 2. dim_doctor
dim_doctor = doctors_cleaned.copy().sort_values('doctor_id').reset_index(drop=True)
dim_doctor.insert(0, 'doctor_key', range(1, len(dim_doctor) + 1))

# 3. dim_clinic
dim_clinic = clinics_raw.copy().sort_values('clinic_id').reset_index(drop=True)
dim_clinic.insert(0, 'clinic_key', range(1, len(dim_clinic) + 1))

# 4. dim_date (2025 calendar year)
dates_2025 = pd.date_range(start='2025-01-01', end='2025-12-31', freq='D')
dim_date = pd.DataFrame({
    'date_key': dates_2025.strftime('%Y%m%d').astype(int),
    'full_date': dates_2025.strftime('%Y-%m-%d'),
    'year': dates_2025.year,
    'quarter': dates_2025.quarter,
    'quarter_name': 'Q' + dates_2025.quarter.astype(str),
    'month': dates_2025.month,
    'month_name': dates_2025.strftime('%B'),
    'day': dates_2025.day,
    'day_name': dates_2025.strftime('%A'),
    'day_of_week': dates_2025.dayofweek + 1,
    'is_weekend': dates_2025.dayofweek.isin([5, 6])
})

# 5. fact_appointment
pat_map = dim_patient.set_index('patient_id')['patient_key'].to_dict()
doc_map = dim_doctor.set_index('doctor_id')['doctor_key'].to_dict()
cli_map = dim_clinic.set_index('clinic_id')['clinic_key'].to_dict()
dt_map = dim_date.set_index('full_date')['date_key'].to_dict()

fact_appointment = pd.DataFrame({
    'patient_key': appointments_cleaned['patient_id'].map(pat_map),
    'doctor_key': appointments_cleaned['doctor_id'].map(doc_map),
    'clinic_key': appointments_cleaned['clinic_id'].map(cli_map),
    'date_key': appointments_cleaned['appointment_date'].map(dt_map),
    'appointment_id': appointments_cleaned['appointment_id'],
    'wait_minutes': appointments_cleaned['wait_minutes'],
    'consultation_minutes': appointments_cleaned['consultation_minutes'],
    'fee': appointments_cleaned['fee']
})

# Export CSV files
dim_patient.to_csv('dim_patient.csv', index=False)
dim_doctor.to_csv('dim_doctor.csv', index=False)
dim_clinic.to_csv('dim_clinic.csv', index=False)
dim_date.to_csv('dim_date.csv', index=False)
fact_appointment.to_csv('fact_appointment.csv', index=False)

print(f"dim_patient.csv:      {len(dim_patient)} rows")
print(f"dim_doctor.csv:       {len(dim_doctor)} rows")
print(f"dim_clinic.csv:       {len(dim_clinic)} rows")
print(f"dim_date.csv:         {len(dim_date)} rows")
print(f"fact_appointment.csv: {len(fact_appointment)} rows")


dim_patient.csv:      300 rows
dim_doctor.csv:       20 rows
dim_clinic.csv:       5 rows
dim_date.csv:         365 rows
fact_appointment.csv: 1500 rows


### Step 9 (Task 4.4): Verify Referential Integrity
**What this cell does:** Checks for missing or orphan foreign keys in the fact table across all dimensions.  
**Expected output:** Confirms 0 orphan keys across patient, doctor, clinic, and date dimensions.


In [9]:
orphan_p = fact_appointment['patient_key'].isnull().sum()
orphan_d = fact_appointment['doctor_key'].isnull().sum()
orphan_c = fact_appointment['clinic_key'].isnull().sum()
orphan_dt = fact_appointment['date_key'].isnull().sum()

print("Orphan check results in fact table:")
print(f"- Patient key orphans: {orphan_p}")
print(f"- Doctor key orphans:  {orphan_d}")
print(f"- Clinic key orphans:  {orphan_c}")
print(f"- Date key orphans:    {orphan_dt}")


Orphan check results in fact table:
- Patient key orphans: 0
- Doctor key orphans:  0
- Clinic key orphans:  0
- Date key orphans:    0
